[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch14-ml-ops/01-kinh-te-van-hanh.ipynb)

# Tiền của vận hành: việc thủ công, lỗi âm thầm và giá của việc quan sát

Chương 14 nhiều lần nhắc rằng vận hành học máy (MLOps) phải tự trả được tiền cho chính nó. Trong
sáu phép nhẩm (napkin math) của chương, bốn phép là phép tính tiền: giờ công của việc huấn luyện
lại thủ công, doanh thu mất vì một lỗi âm thầm, và hai ngân sách theo dõi. Sổ tay này làm lại các
phép tính đó, kiểm từng con số với sách, rồi đẩy mỗi phép đi xa hơn chương một bước: phá một giả
định của sách để xem kết luận còn đứng không, và quét một tham số để xem nó đổi kết luận ở đâu.

Không có gì được đo ở đây, mọi con số đều là số học. Các sổ tay 02 tới 04 mới chạy thí nghiệm.

**Bạn sẽ làm:**
1. cộng dồn giờ công của huấn luyện lại thủ công và của một pipeline tự động, tìm điểm hoà vốn,
   rồi xem chuyện gì xảy ra khi mô hình thay đổi và khi đội có nhiều mô hình;
2. tính chi phí lệch giữa huấn luyện và phục vụ (phương trình 2) và cái giá của một lỗi âm thầm
   theo thời gian phát hiện (napkin math 1.2);
3. đặt cạnh đó một lỗi ồn ào: tổn thất mỗi phút của Knight Capital năm 2012;
4. dựng ngân sách theo dõi một nút học máy (napkin math 1.6), rồi xem số giá trị phân biệt
   (cardinality), tần suất làm mới và việc lấy mẫu dấu vết (napkin math 1.5) đổi nó ra sao;
5. đặt lợi ích cạnh chi phí: tỉ số 5 lần của việc theo dõi, 4.53 lần của một nút học máy.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # thứ tự màu cố định cho mọi hình
XAM = "0.45"


def tien(x):
    """Một số tiền để in: đô la, có dấu phẩy ngăn nghìn."""
    return f"{x:,.0f} đô la" if abs(x) >= 100 else f"{x:,.2f} đô la"


print("sẵn sàng")

## 1. Khi nào tự động hoá hoà vốn

Napkin math 1.1 so hai cách giữ một mô hình luôn mới. Huấn luyện lại thủ công tốn 4 giờ công kỹ sư
mỗi tuần, mãi mãi. Dựng một pipeline tự động tốn 80 giờ công, trả một lần, và trong phép tính đơn
giản hoá của sách, sau đó không tốn giờ định kỳ nào.

**Dự đoán:** sau bao nhiêu tuần thì pipeline rẻ hơn? Và sau 1 năm, đội thủ công đã bỏ ra bao nhiêu
giờ, gấp mấy lần pipeline?

In [ ]:
GIO_THU_CONG = sach(4, nguon="4 giờ mỗi tuần")        # giờ công mỗi tuần, mãi mãi
GIO_DUNG_PIPELINE = sach(80, nguon="80 giờ")            # trả một lần
GIO_PIPELINE_DINH_KY = sach(0, nguon="0 giờ")           # giờ định kỳ của pipeline, theo sách
TUAN_MOT_NAM = 52

tuan = np.arange(TUAN_MOT_NAM + 1)
gio_thu_cong = GIO_THU_CONG * tuan
gio_pipeline = GIO_DUNG_PIPELINE + GIO_PIPELINE_DINH_KY * tuan
hoa_von = GIO_DUNG_PIPELINE / (GIO_THU_CONG - GIO_PIPELINE_DINH_KY)

theo_sach("điểm hoà vốn (tuần)", hoa_von, sach=20, nguon="20 tuần")
print(f"sau {TUAN_MOT_NAM} tuần: thủ công {gio_thu_cong[-1]} giờ, pipeline {gio_pipeline[-1]} giờ,"
      f" gấp {gio_thu_cong[-1] / gio_pipeline[-1]:.1f} lần")

Sách tự chỉ ra cái bẫy của phép tính này: nó giả định mô hình **không bao giờ thay đổi**, trong khi
thêm đặc trưng có thể làm công sức thủ công tăng lên. Chiều ngược lại cũng có: một pipeline thật
vẫn cần người bảo trì. Ô dưới thử hai giả định của sổ tay, không phải của sách: việc thủ công tăng
thêm 1 giờ mỗi tuần sau mỗi quý (13 tuần) vì đặc trưng mới, và pipeline tốn 1 giờ bảo trì mỗi tuần.

In [ ]:
def gio_cong_don(tuan, gio_dau, tang_moi_quy=0.0):
    """Giờ cộng dồn tới mỗi tuần, khi giờ mỗi tuần tăng thêm `tang_moi_quy` sau mỗi 13 tuần."""
    moi_tuan = gio_dau + tang_moi_quy * (tuan // 13)
    return np.concatenate([[0.0], np.cumsum(moi_tuan[1:])])


def tuan_hoa_von(thu_cong, pipeline):
    vuot = np.nonzero(thu_cong >= pipeline)[0]
    return int(vuot[0]) if len(vuot) else None


TANG_MOI_QUY, BAO_TRI = 1.0, 1.0   # giờ mỗi tuần, giả định của sổ tay
KICH_BAN = {
    "sách: mô hình không đổi": (0.0, 0.0),
    "việc thủ công tăng theo quý": (TANG_MOI_QUY, 0.0),
    "pipeline cần bảo trì": (0.0, BAO_TRI),
    "cả hai": (TANG_MOI_QUY, BAO_TRI),
}
duong = {}
for ten, (tang, bao_tri) in KICH_BAN.items():
    tc = gio_cong_don(tuan, GIO_THU_CONG, tang)
    pl = GIO_DUNG_PIPELINE + bao_tri * tuan
    duong[ten] = (tc, pl)
    print(f"{ten:<30} hoà vốn ở tuần {tuan_hoa_von(tc, pl)}, sau 1 năm: thủ công {tc[-1]:.0f} giờ,"
          f" pipeline {pl[-1]:.0f} giờ")
assert tuan_hoa_von(*duong["sách: mô hình không đổi"]) == 20

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
tc, pl = duong["sách: mô hình không đổi"]
ax.plot(tuan, tc, color=MAU[1], lw=2.2, label="thủ công, 4 giờ mỗi tuần (sách)")
ax.plot(tuan, pl, color=MAU[0], lw=2.2, label="pipeline, 80 giờ một lần (sách)")
tc2, pl2 = duong["cả hai"]
ax.plot(tuan, tc2, color=MAU[1], lw=1.4, ls="--", label="thủ công tăng 1 giờ mỗi quý (giả định)")
ax.plot(tuan, pl2, color=MAU[0], lw=1.4, ls="--", label="pipeline bảo trì 1 giờ mỗi tuần (giả định)")
ax.plot([hoa_von], [GIO_DUNG_PIPELINE], "o", color="0.2", ms=7, zorder=5)
ax.annotate("hoà vốn: tuần 20", (hoa_von, GIO_DUNG_PIPELINE), xytext=(14, -34), textcoords="offset points",
            fontsize=8, ha="left", arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.set_xlabel("tuần")
ax.set_ylabel("giờ công cộng dồn")
ax.set_title("Thủ công nhanh lúc đầu, pipeline rẻ hơn về sau")
ax.set_xlim(0, TUAN_MOT_NAM)
ax.set_ylim(0, None)
ax.legend(loc="upper left", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Hai giả định kéo điểm hoà vốn theo hai chiều ngược nhau, nhưng trong cả bốn kịch bản pipeline vẫn
rẻ hơn trước khi hết năm. Phép tính của sách đúng ở hình dạng: chi phí bảo trì cộng dồn suốt vòng
đời, chi phí xây chỉ trả một lần.

### Tự động hoá giải quyết giới hạn năng lực

Nhận xét hệ thống của napkin math 1.1 không nói về tốc độ. Nó nói một đội thủ công sớm muộn tới lúc
việc bảo trì các mô hình đang chạy chặn họ triển khai mô hình mới. Mô phỏng dưới đây dùng hai giả
định của sổ tay: đội có 40 giờ công mỗi tuần cho học máy, và phát triển một mô hình mới tốn 40 giờ.
Mỗi tuần, giờ còn lại sau bảo trì được dồn vào mô hình kế tiếp. Đội thủ công trả 4 giờ mỗi tuần cho
mỗi mô hình đang chạy; đội tự động trả thêm 80 giờ dựng pipeline cho mỗi mô hình, rồi không trả gì.

**Dự đoán:** sau 1 năm, đội nào có nhiều mô hình trong sản xuất hơn? Đội nào dẫn trước sau 1 tháng?

In [ ]:
GIO_DOI_MOI_TUAN, GIO_PHAT_TRIEN = 40.0, 40.0   # giả định của sổ tay


def mo_phong_doi(gio_bao_tri_moi_mo_hinh, gio_dung_moi_mo_hinh, so_tuan=TUAN_MOT_NAM):
    """Số mô hình trong sản xuất sau mỗi tuần."""
    so_mo_hinh, ton, lich_su = 0, 0.0, [0]
    for _ in range(so_tuan):
        ton += max(0.0, GIO_DOI_MOI_TUAN - gio_bao_tri_moi_mo_hinh * so_mo_hinh)
        while ton >= GIO_PHAT_TRIEN + gio_dung_moi_mo_hinh:
            ton -= GIO_PHAT_TRIEN + gio_dung_moi_mo_hinh
            so_mo_hinh += 1
        lich_su.append(so_mo_hinh)
    return np.array(lich_su)


doi_thu_cong = mo_phong_doi(GIO_THU_CONG, 0.0)
doi_tu_dong = mo_phong_doi(GIO_PIPELINE_DINH_KY, GIO_DUNG_PIPELINE)
tran_thu_cong = GIO_DOI_MOI_TUAN / GIO_THU_CONG
assert doi_thu_cong.max() <= tran_thu_cong          # 10 mô hình là ăn hết 40 giờ bảo trì
for t in (4, 13, 26, 52):
    print(f"tuần {t:>2}: thủ công {doi_thu_cong[t]:>2} mô hình, tự động {doi_tu_dong[t]:>2} mô hình")
print(f"trần của đội thủ công: {tran_thu_cong:.0f} mô hình, vì mỗi mô hình ăn {GIO_THU_CONG} giờ mỗi tuần")

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.step(tuan, doi_thu_cong, where="post", color=MAU[1], lw=2.2, label="thủ công")
ax.step(tuan, doi_tu_dong, where="post", color=MAU[0], lw=2.2, label="có pipeline")
ax.axhline(tran_thu_cong, color=XAM, lw=0.9, ls=":")
ax.text(1, tran_thu_cong + 0.3, "trần của đội thủ công: bảo trì ăn hết 40 giờ", fontsize=8, color="0.3")
ax.set_xlabel("tuần")
ax.set_ylabel("số mô hình trong sản xuất")
ax.set_title("Cùng 40 giờ mỗi tuần (giả định): bảo trì thủ công chặn mô hình mới")
ax.set_xlim(0, TUAN_MOT_NAM)
ax.set_ylim(0, doi_tu_dong.max() + 2)
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.legend(loc="lower right", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Đội thủ công dẫn trước trong những tuần đầu, vì nó không trả 80 giờ dựng pipeline: đó là lý do các
nhóm thường ngần ngại đầu tư tự động hoá. Nhưng mỗi mô hình nó thêm vào lại ăn bớt giờ của mô hình
sau, nên đường của nó cong dần rồi nằm ngang ở 10 mô hình. Đội có pipeline đi chậm mà đều, và vượt
qua. Đúng như sách nói: tự động hoá giải quyết giới hạn năng lực, không chỉ tốc độ.

## 2. Chi phí lệch, rồi cái giá của một lỗi âm thầm

Phương trình 2 của chương đo tác động tài chính của sự không nhất quán giữa huấn luyện và phục vụ:

$$\text{chi phí lệch} = \text{tỉ lệ lệch} \times Q \times C_\text{lỗi}$$

tức tỉ lệ truy vấn bị lệch giữa huấn luyện và phục vụ (training-serving skew) ảnh hưởng, nhân tổng
lượng truy vấn trong kỳ kế toán, nhân chi phí kinh doanh trung bình của mỗi dự đoán sai. Ví dụ của
sách: 1,000,000 truy vấn mỗi ngày, 1 phần trăm lỗi do lệch, mỗi lỗi tốn 0.10 đô la.

In [ ]:
NGAY_MOT_NAM = sach(365, trich="28 days/365 days")
TI_LE_LECH = sach(1, nguon="1 phần trăm · 0.10 đô la") / 100
Q_NGAY = sach(1_000_000, nguon="1,000,000 truy vấn mỗi ngày")
C_LOI = sach(0.10, nguon="0.10 đô la")

chi_phi_lech_nam = TI_LE_LECH * Q_NGAY * C_LOI * NGAY_MOT_NAM
theo_sach("chi phí lệch mỗi năm (đô la)", chi_phi_lech_nam, sach=365_000, nguon="365,000 đô la mỗi năm")

# Bảng 29: kho đặc trưng cơ bản tốn tới 20 nghìn đô la mỗi năm. Nó phải chặn bao nhiêu lỗi lệch để hoà vốn?
KHO_DAC_TRUNG_NAM = sach(20e3, nguon="5–20 nghìn")
print(f"kho đặc trưng đắt nhất của bảng 29 hoà vốn khi chặn được {KHO_DAC_TRUNG_NAM / chi_phi_lech_nam:.1%}"
      f" số lỗi lệch, tức khoảng 1 trong {chi_phi_lech_nam / KHO_DAC_TRUNG_NAM:.0f} lỗi")

Một khoản 365,000 đô la mỗi năm từ một tỉ lệ lỗi chỉ 1 phần trăm: đó là lý do sách gọi các cơ chế
nhất quán là những khoản đầu tư có lợi tức đo được.

### Một lỗi âm thầm, và thời gian phát hiện

Napkin math 1.2 đặt một hệ gợi ý sản phẩm đem lại 50 triệu đô la doanh thu mỗi năm. Một lỗi triển
khai gây lệch, làm chất lượng gợi ý giảm 5 phần trăm, và tỉ lệ chuyển đổi giảm theo cùng tỉ lệ. Rà
soát thủ công hằng tháng phát hiện sau khoảng 28 ngày; kiểm tra tự động hằng ngày phát hiện sau 1
ngày. Doanh thu mất là doanh thu năm, nhân mức giảm, nhân phần của năm mà lỗi còn chạy.

**Dự đoán:** một lỗi như thế tốn bao nhiêu khi rà soát hằng tháng? Kiểm tra hằng ngày tiết kiệm bao
nhiêu mỗi năm, với 4 sự cố mỗi năm như kịch bản giả định?

In [ ]:
DOANH_THU = sach(50e6, nguon="50 triệu đô la")
MUC_GIAM = sach(5, nguon="50 triệu đô la · 5 phần trăm") / 100
TTD_THANG = sach(28, nguon="28 ngày")
TTD_NGAY = sach(1, nguon="1 ngày · 6,849.3 đô la")
SU_CO_NAM = sach(4, nguon="4 sự cố")


def ton_that(thoi_gian_phat_hien):
    """Doanh thu mất trong một sự cố kéo dài tới lúc được phát hiện (ngày)."""
    return DOANH_THU * MUC_GIAM * thoi_gian_phat_hien / NGAY_MOT_NAM


theo_sach("tổn thất, rà soát hằng tháng (đô la)", ton_that(TTD_THANG), sach=191_780.8, nguon="191,780.8 đô la")
theo_sach("tổn thất, kiểm tra hằng ngày (đô la)", ton_that(TTD_NGAY), sach=6_849.3, nguon="6,849.3 đô la")
chenh_mot_su_co = ton_that(TTD_THANG) - ton_that(TTD_NGAY)
theo_sach("chênh lệch mỗi sự cố (đô la)", chenh_mot_su_co, sach=184_931.5, nguon="184,931.5 đô la")
theo_sach("tránh được mỗi năm (đô la)", SU_CO_NAM * chenh_mot_su_co, sach=739_726, nguon="739,726 đô la", tol=1)

Sách viết "gần 739,726 đô la"; phép nhân cho 739,726.03, nên phép so ở trên cho phép lệch 1 đô la.

Tổn thất tăng **tuyến tính** theo thời gian phát hiện: mỗi ngày lỗi còn chạy tốn đúng một khoản như
nhau. Hình dưới vẽ đường đó, cùng với khoản chi cho theo dõi và cảnh báo mà bảng 29 của chương ước
lượng, từ 2 tới 10 nghìn đô la mỗi năm.

In [ ]:
THEO_DOI_THAP = sach(2, nguon="2–10 nghìn mỗi năm") * 1e3
THEO_DOI_CAO = sach(10, nguon="2–10 nghìn mỗi năm") * 1e3
ngay = np.linspace(0, 35, 200)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ngay, ton_that(ngay) / 1e3, color=MAU[1], lw=2.2)
for d, ten, lech in ((TTD_NGAY, "hằng ngày", (0, 62)), (7, "hằng tuần (giả định)", (10, -16)),
                     (TTD_THANG, "hằng tháng", (-12, -26))):
    ax.plot([d], [ton_that(d) / 1e3], "o", color=MAU[1], ms=7, mec="white", mew=1.2, zorder=5)
    ax.annotate(f"{ten}: {ton_that(d):,.0f} đô la", (d, ton_that(d) / 1e3), xytext=lech,
                textcoords="offset points", fontsize=8, ha="right" if lech[0] < 0 else "left",
                arrowprops=dict(arrowstyle="-", color="0.6", lw=0.7) if lech[1] > 40 else None)
ax.axhspan(THEO_DOI_THAP / 1e3, THEO_DOI_CAO / 1e3, color=MAU[0], alpha=0.18, lw=0)
ax.text(34.5, THEO_DOI_CAO / 1e3 + 3, "theo dõi và cảnh báo cả năm (bảng 29)", fontsize=8, color="0.25", ha="right")
ax.set_xlabel("thời gian phát hiện (ngày)")
ax.set_ylabel("doanh thu mất trong một sự cố (nghìn đô la)")
ax.set_title("Một lỗi âm thầm: mỗi ngày chưa phát hiện tốn như nhau")
ax.set_xlim(0, 35)
ax.set_ylim(0, None)
plt.tight_layout()
plt.show()
print(f"mỗi ngày chậm phát hiện tốn {ton_that(1):,.1f} đô la;"
      f" theo dõi cả năm theo bảng 29 bằng {THEO_DOI_THAP / ton_that(1):.1f} tới {THEO_DOI_CAO / ton_that(1):.1f} ngày như thế")

Chi phí theo dõi cả năm chỉ ngang vài ngày của một sự cố. Đó là lý do câu hỏi của napkin math 1.2,
"có đáng công không", có đáp án rõ ở mức doanh thu này.

### Một tinh chỉnh: sự cố bắt đầu vào ngày nào?

Phép tính của sách cho rà soát hằng tháng trọn 28 ngày, tức như thể lỗi bắt đầu ngay sau một lần
rà soát. Nếu lỗi bắt đầu vào một ngày bất kỳ, nó phải chờ tới lần rà soát kế tiếp, trung bình nửa
chu kỳ. Ngoài ra, kiểm tra hằng ngày chỉ thấy lỗi khi số liệu đã về: ô dưới thêm 2 ngày chờ nhãn,
một giả định của sổ tay (sổ tay 03 xem việc chờ nhãn kỹ hơn).

In [ ]:
rng = np.random.default_rng(SEED)
bat_dau = rng.uniform(0, TTD_THANG, size=100_000)       # ngày bắt đầu trong một chu kỳ rà soát
CHO_NHAN = 2.0                                          # ngày, giả định của sổ tay
cho_thang = TTD_THANG - bat_dau                         # chờ tới lần rà soát tháng kế tiếp
cho_ngay = np.ceil(bat_dau) - bat_dau + CHO_NHAN        # chờ tới lần kiểm tra ngày kế tiếp, cộng chờ nhãn
print(f"rà soát hằng tháng: chờ trung bình {cho_thang.mean():.1f} ngày, tối đa {cho_thang.max():.1f}")
print(f"kiểm tra hằng ngày, thêm {CHO_NHAN:.0f} ngày chờ nhãn: chờ trung bình {cho_ngay.mean():.1f} ngày")
tiet_kiem_tb = SU_CO_NAM * (ton_that(cho_thang).mean() - ton_that(cho_ngay).mean())
print(f"tránh được mỗi năm, tính trung bình: {tiet_kiem_tb:,.0f} đô la (sách, trường hợp trọn chu kỳ:"
      f" {SU_CO_NAM * chenh_mot_su_co:,.0f})")

Con số trung bình chưa tới một nửa con số của sách, vì sách lấy trường hợp lỗi chạy trọn một chu kỳ
rà soát. Kết luận không đổi: khoản tránh được vẫn lớn hơn chi phí theo dõi của bảng 29 hàng chục lần.

## 3. Một lỗi ồn ào: Knight Capital, 2012

Lỗi âm thầm tốn tiền theo **ngày**. War story 1.1 của chương là kiểu ngược lại: tháng 8 năm 2012,
một lần triển khai cập nhật phần mềm định tuyến lệnh trên 7 trong 8 máy chủ, bỏ sót máy thứ tám. Máy
cũ chạy một vòng lặp không bị chặn, gửi khoảng 1,400 lệnh mỗi giây, và trong 45 phút gây ra hơn 4
triệu giao dịch ngoài ý muốn, lỗ 460 triệu đô la.

In [ ]:
LENH_MOI_GIAY = sach(1_400, nguon="khoảng 1,400 lệnh mỗi giây", tol=50)
PHUT = sach(45, nguon="45 phút · hơn 4 triệu")
LO = sach(460e6, nguon="460 triệu đô la")
GIAO_DICH = sach(4e6, nguon="45 phút · hơn 4 triệu")

lo_moi_phut = LO / PHUT
print(f"lỗ trung bình: {lo_moi_phut / 1e6:.1f} triệu đô la mỗi phút, {lo_moi_phut / 60:,.0f} đô la mỗi giây")
print(f"1,400 lệnh mỗi giây trong 45 phút: {LENH_MOI_GIAY * PHUT * 60 / 1e6:.2f} triệu lệnh;"
      f" sách ghi hơn {GIAO_DICH / 1e6:.0f} triệu giao dịch")
print(f"so với lỗi âm thầm ở phần 2: một phút ở đây bằng {lo_moi_phut / ton_that(1):,.0f} ngày lỗi âm thầm")
for k in (1, 5, 15):
    print(f"  một cầu dao ngắt sau {k:>2} phút: lỗ khoảng {k * lo_moi_phut / 1e6:,.0f} triệu đô la (giả định lỗ đều theo thời gian)")

Hai con số về lệnh không khớp nhau, và cũng không cần khớp: 1,400 mỗi giây là tốc độ gửi **yêu cầu
lệnh mẹ**, còn hơn 4 triệu là số **giao dịch đã khớp**. Một lệnh mẹ có thể sinh nhiều giao dịch, và
sách không cho tỉ lệ ấy, nên sổ tay không ép hai số bằng nhau.

Điều đáng giữ là thang thời gian. Lỗi âm thầm tốn vài nghìn đô la mỗi ngày và cần cả tuần mới lộ ra;
lỗi ồn ào tốn hàng triệu mỗi phút và lộ ra ngay. Vì thế chương dùng hai loại cơ chế khác nhau: theo
dõi thống kê cho loại thứ nhất, và cầu dao (circuit breaker) cùng việc quay lại phiên bản trước
(rollback) tức thì cho loại thứ hai.

## 4. Ngân sách theo dõi một nút học máy

Theo dõi cũng tốn tiền. Phương trình 14 của chương chia chi phí theo dõi thành bốn phần: nạp dữ liệu,
lưu trữ, tính toán cho truy vấn và bảng điều khiển, và đánh giá cảnh báo. Napkin math 1.6 tính ba
phần đầu cho một nút học máy: một mô hình với 3 biến thể triển khai (sản xuất, canary, staging), mỗi
biến thể phát 50 chỉ số, lấy mẫu mỗi 15 giây, giữ 30 ngày; 2 bảng điều khiển, 3 người, làm mới mỗi
5 phút trong 8 giờ mỗi ngày, 22 ngày làm việc mỗi tháng.

**Dự đoán:** trong ba phần nạp, lưu trữ và truy vấn, phần nào lớn nhất?

In [ ]:
BIEN_THE = sach(3, nguon="3 biến thể")
CHI_SO = sach(50, nguon="50 chỉ số")
CHU_KY_MAU = sach(15, nguon="15 giây")                    # giây giữa hai mẫu
NGAY_GIU = sach(30, nguon="30 ngày · 2 bảng điều khiển")
BANG = sach(2, nguon="2 bảng điều khiển")
NGUOI = sach(3, nguon="3 người")
LAM_MOI = sach(5, nguon="5 phút · 25.9 triệu điểm")       # phút giữa hai lần làm mới
GIO_NGAY = sach(8, trich="8 hours/day")
NGAY_LAM = sach(22, trich="22 days")
GIA_NAP = sach(0.30, nguon="0.30 đô la · 7.8") / 1e6      # đô la mỗi điểm dữ liệu
BYTE_DIEM = sach(8, nguon="8 byte")                       # đã nén
GIA_LUU = sach(1, trich="Cost at $1/GB")                  # đô la mỗi GB mỗi tháng
GIA_TRUY_VAN = sach(0.02, nguon="12,672 truy vấn · 0.02")  # đô la mỗi truy vấn

GIAY_MOT_THANG = NGAY_GIU * 24 * 3600
mau_moi_chuoi = GIAY_MOT_THANG / CHU_KY_MAU
diem = BIEN_THE * CHI_SO * mau_moi_chuoi
gb = diem * BYTE_DIEM / 1e9
truy_van = BANG * NGUOI * (60 / LAM_MOI) * GIO_NGAY * NGAY_LAM
ngan_sach = {"nạp": diem * GIA_NAP, "lưu trữ": gb * GIA_LUU, "truy vấn": truy_van * GIA_TRUY_VAN}

theo_sach("điểm dữ liệu mỗi tháng", diem, sach=25.9e6, nguon="25.9 triệu điểm")
theo_sach("nạp mỗi tháng (đô la)", ngan_sach["nạp"], sach=7.8, nguon="0.30 đô la · 7.8")
theo_sach("dung lượng lưu (GB)", gb, sach=0.2, nguon="0.2 GB")
theo_sach("lưu trữ mỗi tháng (đô la)", ngan_sach["lưu trữ"], sach=0.21, nguon="0.2 GB · 0.21")
theo_sach("truy vấn mỗi tháng", truy_van, sach=12_672, nguon="12,672 truy vấn")
theo_sach("truy vấn mỗi tháng (đô la)", ngan_sach["truy vấn"], sach=253.4, nguon="253.4")
tong_thang = sum(ngan_sach.values())
theo_sach("tổng mỗi tháng (đô la)", tong_thang, sach=261.4, nguon="khoảng 261.4 đô la mỗi tháng")
for ten, v in ngan_sach.items():
    print(f"  {ten:<9} {v:>8.2f} đô la  ({v / tong_thang:.1%})")

Truy vấn chiếm gần hết ngân sách: một bảng điều khiển tự làm mới mỗi 5 phút tốn hơn việc nạp và lưu
mọi chỉ số cộng lại. Phần đánh giá cảnh báo và phụ trội của nền tảng chưa được tính, như sách nói.

Phương trình 11 của chương viết phần nạp và phần lưu thành một **tốc độ chi phí**:

$$\text{tốc độ chi phí} \approx N_\text{chuỗi}\, f_\text{mẫu} \left(C_\text{nạp/mẫu} + B_\text{mẫu}\, T_\text{giữ}\, C_\text{lưu/byte·thời gian}\right)$$

với $N_\text{chuỗi}$ là số chuỗi số liệu sau khi khai triển mọi nhãn. Ở napkin math 1.6, mỗi chuỗi
là một chỉ số của một biến thể, nên $N_\text{chuỗi} = 3 \times 50 = 150$. Ô dưới kiểm rằng phương
trình 11 cho đúng tổng của phần nạp và phần lưu ở trên.

In [ ]:
def toc_do_chi_phi(n_chuoi, chu_ky_mau=CHU_KY_MAU, gia_nap=GIA_NAP, gia_luu=GIA_LUU,
                   byte_diem=BYTE_DIEM, thang_giu=NGAY_GIU / 30):
    """Phương trình 11: đô la mỗi tháng cho nạp và lưu, với thời gian giữ tính bằng tháng."""
    mau_moi_thang = n_chuoi * GIAY_MOT_THANG / chu_ky_mau
    return mau_moi_thang * (gia_nap + byte_diem * thang_giu * gia_luu / 1e9)


n_chuoi = BIEN_THE * CHI_SO
assert abs(toc_do_chi_phi(n_chuoi) - (ngan_sach["nạp"] + ngan_sach["lưu trữ"])) < 1e-9
print(f"{n_chuoi} chuỗi, mỗi chuỗi {mau_moi_chuoi:,.0f} mẫu mỗi tháng:"
      f" {toc_do_chi_phi(n_chuoi):.2f} đô la mỗi tháng cho nạp và lưu")

### Số giá trị phân biệt của nhãn: nơi chi phí bùng nổ

Chương nói yếu tố chi phí lớn nhất của hạ tầng theo dõi là **số giá trị phân biệt của nhãn số liệu**
(cardinality): mỗi giá trị của một nhãn nhân số chuỗi lên. Gắn nhãn `vung` (10 giá trị) và
`thiet_bi` (4 giá trị) vào mọi chỉ số là vô hại; gắn `ma_nguoi_dung` thì khác hẳn. Số giá trị của
từng nhãn dưới đây là giả định của sổ tay. Đơn giá lấy theo dải của bảng 21: nạp 0.10 tới 0.50 đô
la mỗi triệu điểm, lưu nhật ký 0.50 tới 2.00 đô la mỗi GB mỗi tháng.

**Dự đoán:** thêm nhãn mã người dùng với 1 triệu người, chi phí nạp và lưu mỗi tháng thành bao nhiêu?

In [ ]:
NAP_THAP = sach(0.10, nguon="0.10–0.50 đô la mỗi triệu điểm") / 1e6
NAP_CAO = sach(0.50, nguon="0.10–0.50 đô la mỗi triệu điểm") / 1e6
LUU_THAP = sach(0.50, nguon="0.50–2.00 mỗi GB tháng")
LUU_CAO = sach(2.00, nguon="0.50–2.00 mỗi GB tháng")

# Mỗi dòng là một bộ nhãn gắn vào mọi chỉ số; số giá trị của từng nhãn là giả định của sổ tay.
NHAN = [("không nhãn (napkin math 1.6)", 1), ("vung (10)", 10), ("vung × thiet_bi (10 × 4)", 10 * 4),
        ("vung × thiet_bi × ma_cua_hang", 10 * 4 * 1_000), ("ma_nguoi_dung (1 triệu)", 1_000_000)]


def gon(x):
    """Một số tiền gọn: nghìn, triệu."""
    if x >= 1e6:
        return f"{x / 1e6:,.1f} triệu"
    if x >= 1e4:
        return f"{x / 1e3:,.0f} nghìn"
    return f"{x:,.0f}"


hang = []
for ten, k in NHAN:
    n = n_chuoi * k
    hang.append((ten, n, toc_do_chi_phi(n, gia_nap=NAP_THAP, gia_luu=LUU_THAP),
                 toc_do_chi_phi(n, gia_nap=NAP_CAO, gia_luu=LUU_CAO)))
    print(f"{ten:<32} {n:>13,} chuỗi  {gon(hang[-1][2]):>12} tới {gon(hang[-1][3]):>12} đô la mỗi tháng")

fig, ax = plt.subplots(figsize=(9, 3.9))
y = np.arange(len(hang))
thap = np.array([h[2] for h in hang])
cao = np.array([h[3] for h in hang])
ax.barh(y, cao - thap, left=thap, color=MAU[0], alpha=0.85, height=0.5)
for i, (ten, n, a, b) in enumerate(hang):
    ax.text(b * 1.6, i, f"{gon(a)} – {gon(b)} đô la", va="center", fontsize=8)
ax.set_xscale("log")
ax.set_yticks(y, [h[0] for h in hang])
ax.invert_yaxis()
ax.set_xlim(thap.min() / 3, cao.max() * 300)
ax.set_xlabel("nạp và lưu mỗi tháng (đô la, thang log), dải đơn giá của bảng 21")
ax.set_title("Mỗi nhãn nhân số chuỗi lên: nhãn mã người dùng là bùng nổ tổ hợp")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

Nhãn vùng và nhãn thiết bị nhân số chuỗi lên 40 lần mà ngân sách vẫn ở mức vài trăm đô la. Thêm mã
cửa hàng thì thành hàng trăm nghìn. Riêng nhãn mã người dùng nhân số chuỗi lên một triệu lần, và chi
phí nạp, lưu thành hàng triệu đô la mỗi tháng, trước cả khi tính truy vấn. Vì thế
chương khuyên lấy mẫu hay gộp các chiều có nhiều giá trị: chúng thường cho khoản tiết kiệm lớn nhất,
ngay lập tức.

### Truy vấn: chi phí tích lại âm thầm

Phần truy vấn tăng theo số người nhân tần suất làm mới. Chương lưu ý mỗi lần tự làm mới đều truy
vấn phía sau, kể cả khi không ai đang xem. Ô dưới quét tần suất làm mới và số người, và thêm một
trường hợp: một tab để mở cả ngày, làm mới 24 giờ mỗi ngày suốt 30 ngày thay vì 8 giờ trong 22 ngày.

In [ ]:
def chi_phi_truy_van(nguoi, lam_moi_phut, gio_ngay=GIO_NGAY, ngay=NGAY_LAM):
    return BANG * nguoi * (60 / lam_moi_phut) * gio_ngay * ngay * GIA_TRUY_VAN


assert abs(chi_phi_truy_van(NGUOI, LAM_MOI) - ngan_sach["truy vấn"]) < 1e-9
cac_lam_moi = [1, 5, 15, 60]
fig, ax = plt.subplots(figsize=(8, 3.9))
for nguoi, mau in zip((3, 10, 30), MAU):
    ax.plot(cac_lam_moi, [chi_phi_truy_van(nguoi, m) for m in cac_lam_moi], "o-", color=mau, lw=2,
            ms=6, label=f"{nguoi} người" + (" (sách)" if nguoi == NGUOI else ""))
ax.plot([LAM_MOI], [ngan_sach["truy vấn"]], "o", ms=11, mfc="none", mec="0.2", mew=1.2)
ax.annotate(f"napkin math 1.6: {ngan_sach['truy vấn']:.1f} đô la", (LAM_MOI, ngan_sach["truy vấn"]),
            xytext=(14, -18), textcoords="offset points", fontsize=8)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xticks(cac_lam_moi, [f"{m} phút" for m in cac_lam_moi])
ax.set_xlabel("khoảng giữa hai lần làm mới bảng điều khiển")
ax.set_ylabel("truy vấn mỗi tháng (đô la, thang log)")
ax.set_title("Chi phí truy vấn tỉ lệ nghịch với khoảng làm mới")
ax.legend(loc="upper right", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()
de_mo = chi_phi_truy_van(NGUOI, LAM_MOI, gio_ngay=24, ngay=30)
print(f"tab để mở 24 giờ suốt 30 ngày: {de_mo:,.1f} đô la mỗi tháng, gấp {de_mo / ngan_sach['truy vấn']:.1f} lần")

Ở napkin math 1.6, giảm tần suất làm mới từ 5 phút xuống 15 phút cắt hai phần ba ngân sách của cả
nút. Đó là lý do sách nói ở quy mô nền tảng, tối ưu chi phí truy vấn ngày càng quan trọng: chi phí
tăng tuyến tính theo số nút giống nhau, nên một thói quen làm mới được nhân lên ở mọi nút.

## 5. Lấy mẫu dấu vết: vì sao không thể đo mọi thứ

Napkin math 1.5 mở đầu rằng "đo mọi thứ" là bất khả thi về vật lý ở quy mô lớn. Ở độ trung thực đầy
đủ, mỗi yêu cầu đều vào dòng số liệu: 1 triệu yêu cầu mỗi giây, mỗi yêu cầu 1 KB. Giữ 1 trên 60 dấu
vết yêu cầu thì lượng nạp trung bình giảm 60 lần; gom mẫu mỗi 60 giây chỉ đổi thời điểm gửi, không
đổi tốc độ byte trung bình. Khuyến nghị của sách là giữ 1 phần trăm yêu cầu thành công và 100 phần
trăm lỗi.

**Dự đoán:** nếu 1 phần trăm yêu cầu là lỗi, chính sách "1 phần trăm thành công, 100 phần trăm lỗi"
giữ nhiều byte hơn hay ít hơn cách giữ 1 trên 60?

In [ ]:
YEU_CAU_GIAY = sach(1e6, nguon="1 triệu yêu cầu mỗi giây")
KB_YEU_CAU = sach(1, nguon="1 KB · 1 GB/s")
MOT_TREN = sach(60, nguon="1 trên 60")
GIU_THANH_CONG = sach(1, nguon="1 phần trăm · 100 phần trăm") / 100
GIU_LOI = sach(100, nguon="1 phần trăm · 100 phần trăm") / 100

day_du = YEU_CAU_GIAY * KB_YEU_CAU * 1e3 / 1e9        # GB/s, 1 KB = 1,000 byte như sách
theo_sach("độ trung thực đầy đủ (GB/s)", day_du, sach=1, nguon="1 GB/s")
mot_tren_60 = day_du / MOT_TREN * 1e3                 # MB/s
theo_sach("giữ 1 trên 60 (MB/s)", mot_tren_60, sach=16.7, nguon="16.7 MB/s")
theo_sach("chênh lệch lượng số liệu (lần)", day_du * 1e3 / mot_tren_60, sach=60, nguon="60 lần")

print("chính sách 1 phần trăm thành công + 100 phần trăm lỗi, theo tỉ lệ lỗi (giả định):")
for ti_le_loi in (0.001, 0.01, 0.05):
    mb_s = day_du * 1e3 * (GIU_THANH_CONG * (1 - ti_le_loi) + GIU_LOI * ti_le_loi)
    print(f"  lỗi {ti_le_loi:>5.1%}: {mb_s:5.1f} MB/s, trong đó lỗi chiếm {GIU_LOI * ti_le_loi * day_du * 1e3 / mb_s:.0%}")

# Một tháng số liệu ở mỗi tốc độ, lưu với dải đơn giá lưu nhật ký của bảng 21
for ten, gb_s in (("đầy đủ", day_du), ("1 trên 60", mot_tren_60 / 1e3)):
    gb_thang = gb_s * GIAY_MOT_THANG
    print(f"{ten:<10} {gb_thang / 1e6:8.3f} PB mỗi tháng → lưu {gb_thang * LUU_THAP:>12,.0f} tới"
          f" {gb_thang * LUU_CAO:>12,.0f} đô la mỗi tháng")

Ở độ trung thực đầy đủ, một tháng dấu vết là vài petabyte: đúng như bảng 18 của sách nói, việc đó
cần một cụm chuyên dụng. Chính sách của sách giữ ít byte hơn cách 1 trên 60 khi lỗi hiếm, nhưng giữ
**trọn** mọi lỗi, tức đúng những yêu cầu cần cho gỡ lỗi. Khi tỉ lệ lỗi tăng, phần lỗi chiếm gần hết
lượng giữ lại, nên một sự cố cũng là lúc chi phí quan sát tăng vọt.

## 6. Lợi ích trên chi phí

Phương trình 15 biện minh việc theo dõi bằng một tỉ số: số sự cố ngăn được nhân chi phí trung bình
mỗi sự cố, chia chi phí theo dõi mỗi năm. Ví dụ của sách: mỗi sự cố tốn trung bình 50,000 đô la,
việc theo dõi ngăn được 5 sự cố mỗi năm, với chi phí theo dõi 50,000 đô la.

Phương trình 16 làm điều tương tự cho cả khoản đầu tư MLOps của một nút: tử số cộng thêm số giờ tự
động hoá tiết kiệm nhân chi phí mỗi giờ. Ví dụ là một mô hình đem lại 1 triệu đô la doanh thu mỗi
năm, tránh được 4 sự cố mỗi năm, mỗi sự cố 25 nghìn đô la, tiết kiệm 20 giờ triển khai mỗi tháng ở
150 đô la mỗi giờ, với khoản đầu tư 30 nghìn đô la mỗi năm.

In [ ]:
SU_CO_NGAN = sach(5, nguon="50,000 đô la · 5 sự cố")
CHI_PHI_SU_CO = sach(50_000, nguon="50,000 đô la · 5 sự cố")
THEO_DOI_NAM = sach(50_000, nguon="50,000 đô la · 5 sự cố")
theo_sach("lợi ích trên chi phí của việc theo dõi (lần)", SU_CO_NGAN * CHI_PHI_SU_CO / THEO_DOI_NAM,
          sach=5, nguon="5 lần")

SU_CO_TRANH = sach(4, nguon="4 sự cố × 25 nghìn")
MOT_SU_CO = sach(25e3, nguon="4 sự cố × 25 nghìn")
GIO_THANG = sach(20, nguon="20 giờ mỗi tháng × 150 đô la")
GIA_GIO = sach(150, nguon="20 giờ mỗi tháng × 150 đô la")
DAU_TU = sach(30e3, nguon="30 nghìn · 4.53 lần")

tranh_su_co = SU_CO_TRANH * MOT_SU_CO
tiet_kiem_gio = GIO_THANG * 12 * GIA_GIO


def ti_so(su_co, gio_thang=GIO_THANG, dau_tu=DAU_TU, mot_su_co=MOT_SU_CO):
    """Phương trình 16."""
    return (su_co * mot_su_co + gio_thang * 12 * GIA_GIO) / dau_tu


theo_sach("sự cố tránh được (đô la)", tranh_su_co, sach=100e3, nguon="= 100 nghìn")
theo_sach("giờ triển khai tiết kiệm (đô la)", tiet_kiem_gio, sach=36e3, nguon="150 đô la = 36 nghìn")
theo_sach("lợi ích trên chi phí của một nút (lần)", ti_so(SU_CO_TRANH), sach=4.53, nguon="4.53 lần")

### Tỉ số đứng vững tới đâu?

Hai tham số đáng thử là số sự cố tránh được và số giờ tiết kiệm. Ô dưới quét cả hai, và đánh dấu
đường hoà vốn, tỉ số bằng 1.

**Dự đoán:** nếu năm nay không có sự cố nào để tránh, khoản đầu tư 30 nghìn đô la có còn hoà vốn?

In [ ]:
su_co = np.arange(0, 9)
fig, ax = plt.subplots(figsize=(8, 3.9))
for gio, mau in zip((20, 10, 5), MAU):
    ax.plot(su_co, ti_so(su_co, gio_thang=gio), "o-", color=mau, lw=2, ms=5,
            label=f"tiết kiệm {gio} giờ mỗi tháng" + (" (sách)" if gio == GIO_THANG else ""))
ax.axhline(1, color=XAM, lw=1, ls="--")
ax.text(8, 1.15, "hoà vốn", fontsize=8, color="0.3", ha="right")
ax.plot([SU_CO_TRANH], [ti_so(SU_CO_TRANH)], "o", ms=11, mfc="none", mec="0.2", mew=1.2)
ax.annotate("4.53 lần (sách)", (SU_CO_TRANH, ti_so(SU_CO_TRANH)), xytext=(-12, 14), textcoords="offset points",
            fontsize=8, ha="right")
ax.set_xlabel("số sự cố tránh được mỗi năm")
ax.set_ylabel("lợi ích trên chi phí (lần)")
ax.set_title(f"Phương trình 16: mỗi sự cố tránh được cộng {MOT_SU_CO / DAU_TU:.2f} vào tỉ số")
ax.set_ylim(0, None)
ax.legend(loc="upper left", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()
print(f"không sự cố nào: tỉ số {ti_so(0):.2f} lần, vì {tiet_kiem_gio:,.0f} đô la tiết kiệm giờ > {DAU_TU:,.0f} đầu tư")
print(f"mỗi sự cố tránh được cộng {MOT_SU_CO / DAU_TU:.2f} vào tỉ số")

Ở ví dụ của sách, riêng phần giờ triển khai tiết kiệm đã lớn hơn khoản đầu tư, nên tỉ số trên 1 kể
cả khi không có sự cố nào. Chỉ khi số giờ tiết kiệm giảm một nửa, việc hoà vốn mới cần tới sự cố tránh
được. Đó là ý của sách: khoản đầu tư xứng đáng không vì công cụ đẹp, mà vì mô hình đủ đắt để việc
ngăn sự cố và rút ngắn triển khai vượt chi phí nền tảng mỗi năm.

### Chi phí mỗi nghìn lần suy luận, để so với chi phí theo dõi

Phương trình 10 đổi chỉ số phần cứng thành một con số có nghĩa với kinh doanh: chi phí thuê GPU mỗi
giờ nhân 1,000, chia thông lượng tính bằng số lần suy luận mỗi giờ. Ví dụ minh hoạ của sách: 3 đô la
mỗi giờ, 50,000 lần suy luận mỗi giờ.

In [ ]:
GIA_GPU_GIO = sach(3, nguon="3 đô la mỗi giờ")
SUY_LUAN_GIO = sach(50_000, nguon="50,000 lần mỗi giờ")


def chi_phi_nghin(suy_luan_gio, gia_gio=GIA_GPU_GIO):
    """Phương trình 10."""
    return gia_gio * 1000 / suy_luan_gio


theo_sach("chi phí mỗi nghìn lần suy luận (đô la)", chi_phi_nghin(SUY_LUAN_GIO), sach=0.06,
          nguon="0.06 đô la mỗi nghìn")
phuc_vu_thang = GIA_GPU_GIO * 24 * NGAY_GIU      # giả định: GPU chạy suốt ngày đêm
print(f"phục vụ một tháng, GPU chạy suốt (giả định): {phuc_vu_thang:,.0f} đô la;"
      f" ngân sách theo dõi ở phần 4 bằng {tong_thang / phuc_vu_thang:.0%} số đó")
for tl in (25_000, 100_000, 200_000):
    print(f"  {tl:>7,} lần mỗi giờ: {chi_phi_nghin(tl):.3f} đô la mỗi nghìn")

Theo dõi một nút tốn khoảng một phần tám tiền thuê GPU của chính nút đó, theo các giả định này.
Chương khuyên theo dõi chi phí mỗi nghìn lần suy luận theo thời gian, vì nó đổi khi giá đổi, khi cơ
cấu tải đổi, hay khi hiệu suất đổi: thông lượng giảm một nửa thì chi phí mỗi lần suy luận tăng gấp
đôi, dù hoá đơn mỗi giờ không đổi.

## Thử thêm

1. Ở phần 1, đặt `TANG_MOI_QUY = 3.0`. Điểm hoà vốn tiến về tuần nào? Rồi đặt `GIO_PHAT_TRIEN = 120`
   trong mô phỏng năng lực: đội có pipeline còn vượt đội thủ công trong năm đầu không?
2. Ở phần 2, hạ `DOANH_THU` xuống 1 triệu đô la, mức của ví dụ phương trình 16. Khoản tránh được mỗi
   năm còn lớn hơn chi phí theo dõi cao nhất của bảng 29 không? Ở doanh thu nào thì hai bên bằng nhau?
3. Ở phần 4, đổi `CHU_KY_MAU` thành 60 và `LAM_MOI` thành 15. Ngân sách của nút giảm còn bao nhiêu,
   và phần nào bây giờ lớn nhất?

## Tóm lại

* Huấn luyện lại thủ công rẻ lúc đầu và đắt về sau: 80 giờ chia 4 giờ mỗi tuần hoà vốn ở tuần 20, và
  lợi thế lớn hơn của tự động hoá là năng lực, vì bảo trì thủ công chặn mô hình mới.
* Một lỗi âm thầm tốn tiền tuyến tính theo thời gian phát hiện; với hệ gợi ý 50 triệu đô la, kiểm tra
  hằng ngày tránh được khoảng 739,726 đô la mỗi năm, hơn nhiều lần chi phí theo dõi.
* Một lỗi ồn ào như Knight Capital tốn hàng triệu đô la mỗi phút, nên cần cầu dao và quay lại phiên bản
  trước tức thì, không phải theo dõi thống kê.
* Theo dõi một nút tốn khoảng 261.4 đô la mỗi tháng, phần lớn là truy vấn; số giá trị phân biệt của
  nhãn số liệu là thứ có thể nhân nó lên hàng triệu lần.
* Tỉ số lợi ích trên chi phí, 5 lần cho việc theo dõi và 4.53 lần cho một nút, chỉ đứng vững khi mô
  hình đủ đắt; hãy quét tham số trước khi tin một tỉ số.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Vận hành học máy — khi hệ thống sẵn sàng mà vẫn sai](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch14-ml-ops/), dựng từ chương
[*ML Operations*](https://mlsysbook.ai/vol1/ml_ops/ml_ops.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.